# Day 7 — Mini-Project: Return Arithmetic You Can Trust

Build the toolkit. Test it. Then answer Q1–Q3. This code follows you for 45
weeks — make it clean.

## Setup

In [ ]:
import os, sys, pathlib
root = pathlib.Path.cwd()
for _ in range(6):
    if (root / "qrc").is_dir():
        break
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 4)
DATA_SOURCE = os.environ.get("QRC_DATA", "real")

from qrc.data import get_prices
from qrc.synth import synthetic_prices

if DATA_SOURCE == "real":
    px = get_prices(["SPY", "XLE"], start="2010-01-01")
    names = ["SPY", "XLE"]
else:
    px = synthetic_prices(n_days=3000, n_assets=2, seed=41, drift_spread=0.0004)
    px.columns = names = ["SPY", "XLE"]

## The toolkit (implement all six)

In [ ]:
def to_log(simple: pd.Series) -> pd.Series:
    """simple returns -> log returns. Fails loudly if any return <= -1."""
    # YOUR CODE

def to_simple(logret: pd.Series) -> pd.Series:
    # YOUR CODE

def total_growth(simple: pd.Series) -> float:
    """Growth of $1 over the whole path: prod(1+r)."""
    # YOUR CODE

def cagr(simple: pd.Series, ppy: int = 252) -> float:
    """Geometric mean return, annualized."""
    # YOUR CODE

def ann_vol(simple: pd.Series, ppy: int = 252) -> float:
    """Annualized volatility: sample std (ddof=1) * sqrt(ppy)."""
    # YOUR CODE

def drawdown(simple: pd.Series) -> pd.Series:
    """The full drawdown series: wealth/cummax(wealth) - 1."""
    # YOUR CODE

## The tests (write these BEFORE trusting outputs)

In [ ]:
r = pd.Series([0.10, -0.05, 0.02])

# T1 hand-computed: growth = 1.10*0.95*1.02 = 1.0659
# assert np.isclose(total_growth(r), 1.10 * 0.95 * 1.02)

# T2 identity: to_simple(to_log(r)) == r
# assert np.allclose(to_simple(to_log(r)), r)

# T3 property: constant 1% daily for 252 days -> CAGR == 1.01**252 - 1
# c = pd.Series([0.01] * 252)
# assert np.isclose(cagr(c), 1.01 ** 252 - 1)

# T4 property: drawdown of a strictly rising series is all zeros
# T5 loud failure: to_log(pd.Series([-1.5])) should raise (impossible return)

Implement T4 and T5 yourself (one line each, roughly). All green before Q1.

In [ ]:
# YOUR CODE for T4, T5, then run all tests

## Q1 — The drag, measured

For both assets: annualized arithmetic mean, CAGR, measured gap, σ²/2
approximation. Which pays more drag? Does the approximation hold?

In [ ]:
rows = []
for name in names:
    s = px[name].pct_change().dropna()
    # YOUR CODE: append the four numbers
report = pd.DataFrame(rows, index=names)
report

## Q2 — The frequency illusion

From the same prices: monthly returns (resample prices to month-end, then
returns). Compute (a) CAGR by compounding monthly returns; (b) "annual
return" by annualizing the mean *daily* return (×252). Same? Which is
growth, which is an expectation? Report the gap in pp/year for both assets.

In [ ]:
# YOUR CODE

## Q3 — Volatility scaling check

Annualized vol from daily returns (×√252) vs from monthly (×√12). Report
both for both assets. Why can they differ? (One sentence — vol clustering
is the keyword; module 09 formalizes.)

In [ ]:
# YOUR CODE

## Reflection

- Which trap is most tempting for YOU?
- Which toolkit test caught a bug during development? (If none: you didn't
  test hard enough — write one more test that tries to break your code.)

In [ ]:
# Your answers:

## Hints

- cagr: years = len(s) / ppy; growth ** (1/years) - 1.
- Q2: monthly CAGR = (prod of monthly growth) ** (12/n_months) - 1.
- Q3: variance of a sum of dependent terms ≠ sum of variances; squared daily
  returns are positively autocorrelated (vol clustering), so daily-based
  vol understates/overstates monthly-based vol depending on the regime mix.